In [1]:
from pathlib import Path
from datasets import load_from_disk
from transformers import AutoTokenizer
import pandas as pd
import numpy as np
import re

SEED = 42
MODEL_NAME = "google/flan-t5-small"
MAX_INPUT_LENGTH = 512

project_dir = Path.cwd()

if project_dir.name == "notebooks":
    project_dir = project_dir.parent

reports_dir = project_dir / "reports"

dataset = load_from_disk(
    str(project_dir / "data" / "healthbrief_sampled")
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

PROMPT = (
    "Summarize the following biomedical research article "
    "in plain language for a general audience:\n\n"
)

print("Dataset:", dataset)
print("Prompt tokens:", len(tokenizer.encode(PROMPT)))
print("Section headings:", dataset["train"][0]["section_headings"])
print("\nArticle:\n", dataset["train"][0]["article"][:1500])

/home/pasali/anaconda3/envs/myenv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset: DatasetDict({
    train: Dataset({
        features: ['article', 'summary', 'section_headings', 'keywords', 'year', 'title'],
        num_rows: 500
    })
    validation: Dataset({
        features: ['article', 'summary', 'section_headings', 'keywords', 'year', 'title'],
        num_rows: 100
    })
    test: Dataset({
        features: ['article', 'summary', 'section_headings', 'keywords', 'year', 'title'],
        num_rows: 100
    })
})
Prompt tokens: 20
Section headings: ['Abstract', 'Introduction', 'Results', 'Discussion', 'Materials', 'and', 'methods']

Article:
 How long neural information is stored in a local brain area reflects functions of that region and is often estimated by the magnitude of the autocorrelation of intrinsic neural signals in the area . Here , we investigated such intrinsic neural timescales in high-functioning adults with autism and examined whether local brain dynamics reflected their atypical behaviours . By analysing resting-state fMRI data , we

In [11]:
def prepare_source_text(article, section_headings, tokenizer, max_length):
    headings = [
        "Abstract", "Introduction", "Results",
        "Discussion", "Conclusion", "Conclusions",
        "Results and Discussion", "Materials and Methods",
        "Methods", "References", "Acknowledgements"
    ]

    if section_headings is not None:
        headings.extend(section_headings)

    headings = {h.strip().lower() for h in headings}

    paragraphs = [
        line.strip()
        for line in article.splitlines()
        if line.strip()
    ]

    sections = {}
    current_section = None
    opening = []

    for paragraph in paragraphs:
        heading, separator, content = paragraph.partition(":")
        heading = heading.strip().lower()

        if heading in headings:
            current_section = heading
            sections.setdefault(heading, [])

            if content.strip():
                sections[heading].append(content.strip())

        elif current_section is None:
            opening.append(paragraph)

        else:
            sections[current_section].append(paragraph)

    sections = {
        name: "\n".join(texts)
        for name, texts in sections.items()
        if texts
    }

    selected = []
    fallback_used = False

    if "abstract" in sections:
        selected.append(("Abstract", sections["abstract"]))
    else:
        opening_text = opening[0] if opening else paragraphs[0]
        selected.append(("Opening (fallback)", opening_text))
        fallback_used = True

    secondary_names = [
        "conclusion",
        "conclusions",
        "discussion",
        "results and discussion",
        "results"
    ]

    secondary_name = next(
        (name for name in secondary_names if name in sections),
        None
    )

    if secondary_name:
        selected.append(
            (secondary_name.title(), sections[secondary_name])
        )

        if secondary_name == "results":
            fallback_used = True

    else:
        fallback_used = True

        keywords = [
            "we found",
            "we show",
            "we demonstrated",
            "we observed",
            "our findings",
            "our results",
            "these results",
            "these findings",
            "taken together",
            "in conclusion"
        ]

        best_paragraph = None
        best_score = 0

        for paragraph in paragraphs[1:]:
            if len(paragraph.split()) < 40:
                continue

            if paragraph in selected[0][1]:
                continue

            score = sum(
                keyword in paragraph.lower()
                for keyword in keywords
            )

            if score > best_score:
                best_score = score
                best_paragraph = paragraph

        if best_paragraph is not None:
            selected.append(
                ("Result-cue paragraph (fallback)", best_paragraph)
            )

    if not sections and len(selected) == 1:
        selected = [("Opening (fallback)", article)]

    def count_tokens(text, special_tokens=False):
        return len(tokenizer.encode(
            text,
            add_special_tokens=special_tokens,
            verbose=False
        ))

    def truncate_text(text, limit):
        if limit <= 0:
            return ""

        offsets = tokenizer(
            text,
            add_special_tokens=False,
            return_offsets_mapping=True,
            verbose=False
        )["offset_mapping"]

        if len(offsets) <= limit:
            return text

        return text[:offsets[limit - 1][1]].strip()

    prompt_tokens = count_tokens(PROMPT, True)
    budget = max_length - prompt_tokens

    if budget <= 0:
        raise ValueError("Insufficient token budget.")

    lengths = [
        count_tokens(text)
        for _, text in selected
    ]

    if len(selected) == 2:
        first = min(lengths[0], budget // 2)
        second = min(lengths[1], budget - first)
        first = min(lengths[0], budget - second)

        limits = [first, second]

    else:
        limits = [min(lengths[0], budget)]

    while True:
        retained = [
            truncate_text(text, limit)
            for (_, text), limit in zip(selected, limits)
        ]

        source_text = "\n\n".join(
            text for text in retained if text
        )

        input_text = PROMPT + source_text
        input_tokens = count_tokens(input_text, True)

        if input_tokens <= max_length:
            break

        limits[limits.index(max(limits))] -= 1

    return {
        "input_text": input_text,
        "source_sections": [
            name for (name, _), text in zip(selected, retained)
            if text
        ],
        "verified_headings": list(sections.keys()),
        "original_token_count": count_tokens(article),
        "retained_token_count": count_tokens(source_text),
        "prompt_token_count": prompt_tokens,
        "input_token_count": input_tokens,
        "truncated": source_text != article.strip(),
        "fallback_used": fallback_used
    }


rng = np.random.default_rng(SEED)

selected_ids = sorted(
    rng.choice(len(dataset["train"]), size=10, replace=False)
)

results = []
prepared_sources = {}

for article_id in selected_ids:
    article_id = int(article_id)
    sample = dataset["train"][article_id]

    result = prepare_source_text(
        sample["article"],
        sample["section_headings"],
        tokenizer,
        MAX_INPUT_LENGTH
    )

    prepared_sources[article_id] = result

    results.append({
        "article_id": article_id,
        "original_tokens": result["original_token_count"],
        "selected_sections": ", ".join(result["source_sections"]),
        "verified_headings": ", ".join(result["verified_headings"]),
        "retained_tokens": result["retained_token_count"],
        "prompt_tokens": result["prompt_token_count"],
        "input_tokens": result["input_token_count"],
        "truncated": result["truncated"],
        "fallback": result["fallback_used"]
    })

selection_df = pd.DataFrame(results)

selection_df["retention_rate"] = (
    selection_df["retained_tokens"] /
    selection_df["original_tokens"] * 100
).round(2)

display(selection_df)

assert len(selection_df) == 10
assert selection_df["article_id"].nunique() == 10
assert selection_df["input_tokens"].between(1, MAX_INPUT_LENGTH).all()
assert selection_df["retained_tokens"].gt(0).all()

reports_dir.mkdir(parents=True, exist_ok=True)

selection_df.to_csv(
    reports_dir / "source_selection_analysis.csv",
    index=False
)

print("Total articles:", len(selection_df))
print("Fallback count:", selection_df["fallback"].sum())
print("Maximum input length:", selection_df["input_tokens"].max())

print("\nSelected sections:")
print(selection_df["selected_sections"].value_counts())

print("\nAverage retention rate:")
print(selection_df["retention_rate"].mean().round(2))

print("\nExample model input:")
print(prepared_sources[int(selected_ids[0])]["input_text"])

,article_id,original_tokens,selected_sections,verified_headings,retained_tokens,prompt_tokens,input_tokens,truncated,fallback,retention_rate
0,42,11344,"Opening (fallback), Result-cue paragraph (fall...",,492,20,512,True,True,4.34
1,43,21903,"Opening (fallback), Result-cue paragraph (fall...",,492,20,512,True,True,2.25
2,47,9523,"Opening (fallback), Result-cue paragraph (fall...",,492,20,512,True,True,5.17
3,100,18238,"Opening (fallback), Result-cue paragraph (fall...",,492,20,512,True,True,2.70
4,214,12195,"Opening (fallback), Result-cue paragraph (fall...",,492,20,512,True,True,4.03
5,216,17163,"Opening (fallback), Result-cue paragraph (fall...",,492,20,512,True,True,2.87
6,322,11842,"Opening (fallback), Result-cue paragraph (fall...",,492,20,512,True,True,4.15
7,347,16227,"Opening (fallback), Result-cue paragraph (fall...",,492,20,512,True,True,3.03
8,380,22365,"Opening (fallback), Result-cue paragraph (fall...",,492,20,512,True,True,2.20
9,425,17185,"Opening (fallback), Result-cue paragraph (fall...",,492,20,512,True,True,2.86


Total articles: 10
Fallback count: 10
Maximum input length: 512

Selected sections:
selected_sections
Opening (fallback), Result-cue paragraph (fallback)    10
Name: count, dtype: int64

Average retention rate:
3.36

Example model input:
Summarize the following biomedical research article in plain language for a general audience:

The dynamics of predator-prey pursuit appears complex , making the development of a framework explaining predator and prey strategies problematic . We develop a model for terrestrial , cursorial predators to examine how animal mass modulates predator and prey trajectories and affects best strategies for both parties . We incorporated the maximum speed-mass relationship with an explanation of why larger animals should have greater turn radii; the forces needed to turn scale linearly with mass whereas the maximum forces an animal can exert scale to a 2/3 power law . This clarifies why in a meta-analysis , we found a preponderance of predator/prey mass ratios th

In [13]:
from datasets import load_dataset

DATASET_NAME = "BioLaySumm/BioLaySumm2025-eLife"
DATASET_REVISION = "144e9c785e3a309da804eb6786feb38e6596f390"


original_validation = load_dataset(
    path=DATASET_NAME,
    revision=DATASET_REVISION,
    split="validation" , 
    streaming=False
)

print(len(original_validation))


Generating test split: 100%|██████████| 142/142 [00:00<00:00, 3936.62 examples/s]

241


In [ ]:
import hashlib

def get_article(article):
    normalized = " ".join(article.lower().split())
    article_id = hashlib.sha256(normalized.encode("utf-8")).hexdigest()
    return article_id



True


In [19]:
train_ids = {get_article(words) for words in dataset["train"]["article"]}

validation_ids = {get_article(words) for words in dataset["validation"]["article"]}

original_validation_ids = {get_article(words) for words in original_validation["article"]}

print("Train IDs:", len(train_ids))
print("Validation IDs:", len(validation_ids))
print("Original Validation IDs:", len(original_validation_ids))

Train IDs: 500
Validation IDs: 100
Original Validation IDs: 241


In [20]:
validation_subset = validation_ids.issubset(original_validation_ids)

train_validation_overlap = train_ids.intersection(validation_ids)

remaining_ids = original_validation_ids.difference(validation_ids)

candidate_ids = remaining_ids.difference(train_ids)

print("Validation subset:", validation_subset)
print("Train-validation overlap:", len(train_validation_overlap))
print("Remaining validation IDs:", len(remaining_ids))
print("Holdout candidate IDs:", len(candidate_ids))

Validation subset: True
Train-validation overlap: 0
Remaining validation IDs: 141
Holdout candidate IDs: 141


In [22]:
holdout_candidates = original_validation.filter(
    lambda row: (
        get_article(row["article"]) in candidate_ids
        and isinstance(row["summary"], str)
        and bool(row["summary"].strip())
    )
)

print("Valid holdout candidates:", len(holdout_candidates))

Filter: 100%|██████████| 241/241 [00:00<00:00, 630.90 examples/s]

Valid holdout candidates: 141


In [25]:
holdout_df = holdout_candidates.to_pandas()

final_holdout_df = holdout_df.sample(
    n=100,
    random_state=42,
    replace=False
).reset_index(drop=True)

final_holdout_df["article_id"] = final_holdout_df["article"].apply(get_article)

print("Final holdout samples:", len(final_holdout_df))
print("Unique article IDs:", final_holdout_df["article_id"].nunique())

Final holdout samples: 100
Unique article IDs: 100


In [30]:
final_holdout_ids = set(final_holdout_df["article_id"])

train_holdout_overlap = train_ids.intersection(final_holdout_ids)

validation_holdout_overlap = validation_ids.intersection(final_holdout_ids)

print("Final holdout IDs:", len(final_holdout_ids))
print("Train-validation overlap:", len(train_validation_overlap))
print("Train-holdout overlap:", len(train_holdout_overlap))
print("Validation-holdout overlap:", len(validation_holdout_overlap))


assert len(train_validation_overlap) == 0
assert len(train_holdout_overlap) == 0
assert len(validation_holdout_overlap) == 0

Final holdout IDs: 100
Train-validation overlap: 0
Train-holdout overlap: 0
Validation-holdout overlap: 0


In [31]:
splits = {
    "train": dataset["train"].to_pandas(),
    "validation": dataset["validation"].to_pandas(),
    "final_holdout": final_holdout_df,
    "original_test": dataset["test"].to_pandas()
}


manifest_records = []

for split_name, split_df in splits.items():
    for _, row in split_df.iterrows():
        manifest_records.append({
            "article_id": get_article(row["article"]),
            "split": split_name,
            "reference_available": isinstance(row["summary"], str) and bool(row["summary"].strip())
        })



manifest_df = pd.DataFrame(manifest_records)

display(manifest_df.head())

print("Total records:", len(manifest_df))

display(manifest_df.groupby("split")["reference_available"].agg(["count", "sum"]))

,article_id,split,reference_available
0,b483e0bb27726037f6415bec466c6177bfc25b746a7ef4...,train,True
1,799313e32c12381642a57c302a075c276a99e408e01009...,train,True
2,829b14a00dc88c3a6c0780b40eee7b7a88d7da63f0ac8d...,train,True
3,5a0d2ce3031eea7cb10188d1fc4c1344adaba0cfa3f18d...,train,True
4,e823290087f6ffb6e3953f646c8cc3e746590636e206f2...,train,True


Total records: 800


,count,sum
split,,
final_holdout,100,100
original_test,100,0
train,500,500
validation,100,100


In [32]:
from datasets import Dataset

manifest_df.to_csv(
    reports_dir / "dataset_split_manifest.csv",
    index=False
)

final_holdout = Dataset.from_pandas(
    final_holdout_df,
    preserve_index=False
)

final_holdout.save_to_disk(
    str(project_dir / "data" / "final_holdout")
)

print("Manifest saved successfully.")
print("Final holdout saved successfully.")

Saving the dataset (1/1 shards): 100%|██████████| 100/100 [00:00<00:00, 10262.80 examples/s]

Manifest saved successfully.
Final holdout saved successfully.


In [40]:
sample = dataset["train"][0]

sample

{'article': 'How long neural information is stored in a local brain area reflects functions of that region and is often estimated by the magnitude of the autocorrelation of intrinsic neural signals in the area . Here , we investigated such intrinsic neural timescales in high-functioning adults with autism and examined whether local brain dynamics reflected their atypical behaviours . By analysing resting-state fMRI data , we identified shorter neural timescales in the sensory/visual cortices and a longer timescale in the right caudate in autism . The shorter intrinsic timescales in the sensory/visual areas were correlated with the severity of autism , whereas the longer timescale in the caudate was associated with cognitive rigidity . These observations were confirmed from neurodevelopmental perspectives and replicated in two independent cross-sectional datasets . Moreover , the intrinsic timescale was correlated with local grey matter volume . This study shows that functional and stru

In [39]:
sample = dataset["train"][0]

result = prepare_source_text(
    article=sample["article"],
    section_headings=sample["section_headings"],
    tokenizer=tokenizer,
    max_length=MAX_INPUT_LENGTH
)

result

{'input_text': 'Summarize the following biomedical research article in plain language for a general audience:\n\nHow long neural information is stored in a local brain area reflects functions of that region and is often estimated by the magnitude of the autocorrelation of intrinsic neural signals in the area . Here , we investigated such intrinsic neural timescales in high-functioning adults with autism and examined whether local brain dynamics reflected their atypical behaviours . By analysing resting-state fMRI data , we identified shorter neural timescales in the sensory/visual cortices and a longer timescale in the right caudate in autism . The shorter intrinsic timescales in the sensory/visual areas were correlated with the severity of autism , whereas the longer timescale in the caudate was associated with cognitive rigidity . These observations were confirmed from neurodevelopmental perspectives and replicated in two independent cross-sectional datasets . Moreover , the intrinsi

In [41]:
target_summary = sample["summary"]

target_token_count = len(
    tokenizer.encode(
        target_summary,
        truncation=False,
        verbose=False
    )
)

print("Source tokens:", result["input_token_count"])
print("Target tokens:", target_token_count)

Source tokens: 512
Target tokens: 466


In [42]:
training_record = {
    "article_id": get_article(sample["article"]),
    "input_text": result["input_text"],
    "target_summary": target_summary,
    "source_token_count": result["input_token_count"],
    "target_token_count": target_token_count,
    "source_sections": result["source_sections"]
}

display(training_record)

{'article_id': 'b483e0bb27726037f6415bec466c6177bfc25b746a7ef4e4d11661dcc422b2ec',
 'input_text': 'Summarize the following biomedical research article in plain language for a general audience:\n\nHow long neural information is stored in a local brain area reflects functions of that region and is often estimated by the magnitude of the autocorrelation of intrinsic neural signals in the area . Here , we investigated such intrinsic neural timescales in high-functioning adults with autism and examined whether local brain dynamics reflected their atypical behaviours . By analysing resting-state fMRI data , we identified shorter neural timescales in the sensory/visual cortices and a longer timescale in the right caudate in autism . The shorter intrinsic timescales in the sensory/visual areas were correlated with the severity of autism , whereas the longer timescale in the caudate was associated with cognitive rigidity . These observations were confirmed from neurodevelopmental perspectives a

| Alan | Neden gerekli? |
|---|---|
| `split` | Makalenin train veya validation verisine ait olduğunu bilmek için. |
| `article_id` | Makaleyi benzersiz biçimde takip etmek için. |
| `input_text` | Modelin okuyacağı metni saklamak için. |
| `target_summary` | Modelin öğrenmesini istediğimiz doğru özeti saklamak için. |
| `source_token_count` | Model girdisinin uzunluğunu kontrol etmek için. |
| `target_token_count` | Hedef özetin uzunluğunu analiz etmek için. |
| `source_sections` | Makalenin hangi bölümlerini seçtiğimizi takip etmek için. |

In [43]:
training_splits = ["train", "validation"]

prepared_records = []

for split in training_splits:
    for sample in dataset[split]:

        result = prepare_source_text(
            article=sample["article"],
            section_headings=sample["section_headings"],
            tokenizer=tokenizer,
            max_length=MAX_INPUT_LENGTH
        )

        target_summary = sample["summary"]

        target_token_count = len(
            tokenizer.encode(
                target_summary,
                truncation=False,
                verbose=False
            )
        )

        prepared_records.append({
            "split": split,
            "article_id": get_article(sample["article"]),
            "input_text": result["input_text"],
            "target_summary": target_summary,
            "source_token_count": result["input_token_count"],
            "target_token_count": target_token_count,
            "source_sections": result["source_sections"]
        })

prepared_df = pd.DataFrame(prepared_records)

print("Total samples:", len(prepared_df))

display(prepared_df.groupby("split").size())

Total samples: 600


split
train         500
validation    100
dtype: int64

In [45]:
target_lengths = [256, 512, 768]

quality_records = []

for split in training_splits:

    split_df = prepared_df[prepared_df["split"] == split]

    lengths = split_df["target_token_count"]

    for max_length in target_lengths:

        total_samples = len(lengths)

        truncated_samples = (lengths > max_length).sum()

        truncation_rate = (truncated_samples / total_samples) * 100

        token_loss = (lengths - max_length ).clip(lower=0)

        mean_token_loss_rate = ((token_loss / lengths) * 100).mean()

        quality_records.append({
            "split": split,
            "max_target_length": max_length,
            "total_samples": total_samples,
            "truncated_samples": truncated_samples,
            "truncation_rate": truncation_rate,
            "mean_token_loss_rate": mean_token_loss_rate
        })

quality_df = pd.DataFrame(quality_records)

display(quality_df)

,split,max_target_length,total_samples,truncated_samples,truncation_rate,mean_token_loss_rate
0,train,256,500,500,100.0,49.180867
1,train,512,500,260,52.0,6.740734
2,train,768,500,4,0.8,0.030313
3,validation,256,100,100,100.0,48.809394
4,validation,512,100,48,48.0,7.362712
5,validation,768,100,0,0.0,0.000000


In [46]:
MAX_TARGET_LENGTH = 768

quality_df.to_csv(
    reports_dir / "training_data_quality.csv",
    index=False
)

print("Training data quality report saved successfully.")

Training data quality report saved successfully.


In [47]:
from datasets import Dataset, DatasetDict

train_df = prepared_df[prepared_df["split"] == "train"]

validation_df = prepared_df[prepared_df["split"] == "validation"]


prepared_dataset = DatasetDict({
    "train": Dataset.from_pandas(
        train_df,
        preserve_index=False
    ),
    "validation": Dataset.from_pandas(
        validation_df , 
        preserve_index=False
    )
})

display(prepared_dataset)

DatasetDict({
    train: Dataset({
        features: ['split', 'article_id', 'input_text', 'target_summary', 'source_token_count', 'target_token_count', 'source_sections'],
        num_rows: 500
    })
    validation: Dataset({
        features: ['split', 'article_id', 'input_text', 'target_summary', 'source_token_count', 'target_token_count', 'source_sections'],
        num_rows: 100
    })
})

In [48]:
prepared_dataset.save_to_disk(
    str(project_dir / "data" / "healthbrief_prepared")
)

print("Prepared dataset saved successfully.")

Saving the dataset (1/1 shards): 100%|██████████| 100/100 [00:00<00:00, 35620.42 examples/s]

Prepared dataset saved successfully.


In [49]:
from datasets import load_from_disk

assert set(prepared_dataset.keys()) == {"train", "validation"}

assert len(prepared_dataset["train"]) == 500

assert len(prepared_dataset["validation"]) == 100

assert prepared_df["source_token_count"].between(1, MAX_INPUT_LENGTH).all()

assert prepared_df["target_token_count"].gt(0).all()

print("Basic dataset validation: PASS")


assert prepared_df["input_text"].apply(
    lambda text: isinstance(text, str) and bool(text.strip())
).all()

assert prepared_df["target_summary"].apply(
    lambda text: isinstance(text, str) and bool(text.strip())
).all()

print("Text validity: PASS")


assert prepared_df["article_id"].is_unique

print("Duplicate control: PASS")


prepared_ids = set(prepared_df["article_id"])

assert prepared_ids.isdisjoint(final_holdout_ids)

assert prepared_ids.isdisjoint(
    set(manifest_df.loc[
        manifest_df["split"] == "original_test", "article_id"
    ])
)

print("Holdout and test isolation: PASS")


for split in training_splits:

    expected_ids = set(
        manifest_df.loc[
            manifest_df["split"] == split, "article_id"
        ]
    )

    actual_ids = set(prepared_dataset[split]["article_id"])

    assert actual_ids == expected_ids

print("Manifest consistency: PASS")


reloaded_dataset = load_from_disk(
    str(project_dir / "data" / "healthbrief_prepared")
)

assert set(reloaded_dataset.keys()) == {"train", "validation"}

for split in training_splits:
    assert reloaded_dataset[split].to_pandas().equals(
        prepared_dataset[split].to_pandas()
    )

print("Saved dataset validation: PASS")

print("\nAll dataset integrity tests passed.")

Basic dataset validation: PASS
Text validity: PASS
Duplicate control: PASS
Holdout and test isolation: PASS
Manifest consistency: PASS
Saved dataset validation: PASS

All dataset integrity tests passed.


In [50]:
from transformers import AutoModelForSeq2SeqLM

MODEL_NAME = "google/flan-t5-small"

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

display(model.config)

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 2856.79it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


T5Config {
  "architectures": [
    "T5ForConditionalGeneration"
  ],
  "classifier_dropout": 0.0,
  "d_ff": 1024,
  "d_kv": 64,
  "d_model": 512,
  "decoder_start_token_id": 0,
  "dense_act_fn": "gelu_new",
  "dropout_rate": 0.1,
  "dtype": "float32",
  "eos_token_id": 1,
  "feed_forward_proj": "gated-gelu",
  "initializer_factor": 1.0,
  "is_decoder": false,
  "is_encoder_decoder": true,
  "is_gated_act": true,
  "layer_norm_epsilon": 1e-06,
  "model_type": "t5",
  "n_positions": 512,
  "num_decoder_layers": 8,
  "num_heads": 6,
  "num_layers": 8,
  "output_past": true,
  "pad_token_id": 0,
  "relative_attention_max_distance": 128,
  "relative_attention_num_buckets": 32,
  "scale_decoder_outputs": false,
  "task_specific_params": {
    "summarization": {
      "early_stopping": true,
      "length_penalty": 2.0,
      "max_length": 200,
      "min_length": 30,
      "no_repeat_ngram_size": 3,
      "num_beams": 4,
      "prefix": "summarize: "
    },
    "translation_en_to_de": {
   

In [51]:
architecture = {
    "model_architecture": model.config.architectures[0],
    "encoder_layers": model.config.num_layers,
    "decoder_layers": model.config.num_decoder_layers,
    "hidden_size": model.config.d_model,
    "attention_heads": model.config.num_heads,
    "vocabulary_size": model.config.vocab_size,
    "feed_forward_size": model.config.d_ff,
    "key_value_size": model.config.d_kv,
    "feed_forward_type": model.config.feed_forward_proj
}

display(pd.DataFrame(
    architecture.items(),
    columns=["feature", "value"]
))

,feature,value
0,model_architecture,T5ForConditionalGeneration
1,encoder_layers,8
2,decoder_layers,8
3,hidden_size,512
4,attention_heads,6
5,vocabulary_size,32128
6,feed_forward_size,1024
7,key_value_size,64
8,feed_forward_type,gated-gelu


| Yöntem | Temel amacı |
|---|---|
| LoRA | Eğitilebilir parametre sayısını azaltmak |
| Quantization | Ağırlıkları saklamak için gereken bit sayısını azaltmak |
| QLoRA | Quantization ve LoRA yöntemlerini birlikte kullanmak |

LoRA uyguladığımızda temel model ağırlıkları bellekte kalmaya devam eder. Quantization uyguladığımızda ise ağırlıkların bellekte kapladığı alan azaltılabilir.

In [53]:
total_parameters = sum(
    p.numel() for p in model.parameters()
)

trainable_parameters = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

model_dtype = next(model.parameters()).dtype

weight_memory_bytes = sum(
    p.numel() * p.element_size() for p in model.parameters()
)

weight_memory_mib = weight_memory_bytes / (1024 ** 2)

print("Total parameters:", total_parameters)
print("Trainable parameters:", trainable_parameters)
print("Model dtype:", model_dtype)
print("Weight memory (MiB):", weight_memory_mib)

Total parameters: 76961152
Trainable parameters: 76961152
Model dtype: torch.float32
Weight memory (MiB): 293.58349609375


| Katman | Görevi |
|---|---|
| Encoder Self-Attention | Makaledeki tokenların birbirleriyle ilişkisini öğrenir. |
| Decoder Self-Attention | Hedef dizideki önceki tokenlar arasındaki ilişkileri işler. |
| Cross-Attention | Decoder'ın kaynak makalenin encoder temsillerinden yararlanmasını sağlar. |
| Feed-Forward | Her tokenın temsilini doğrusal olmayan dönüşümlerle geliştirir. |

In [55]:
for name, module in model.named_modules():
    if name.startswith("encoder.block.0.") or name.startswith("decoder.block.0."):
        print(name, type(module).__name__)

encoder.block.0.layer ModuleList
encoder.block.0.layer.0 T5LayerSelfAttention
encoder.block.0.layer.0.SelfAttention T5Attention
encoder.block.0.layer.0.SelfAttention.q Linear
encoder.block.0.layer.0.SelfAttention.k Linear
encoder.block.0.layer.0.SelfAttention.v Linear
encoder.block.0.layer.0.SelfAttention.o Linear
encoder.block.0.layer.0.SelfAttention.relative_attention_bias Embedding
encoder.block.0.layer.0.layer_norm T5LayerNorm
encoder.block.0.layer.0.dropout Dropout
encoder.block.0.layer.1 T5LayerFF
encoder.block.0.layer.1.DenseReluDense T5DenseGatedActDense
encoder.block.0.layer.1.DenseReluDense.wi_0 Linear
encoder.block.0.layer.1.DenseReluDense.wi_1 Linear
encoder.block.0.layer.1.DenseReluDense.wo Linear
encoder.block.0.layer.1.DenseReluDense.dropout Dropout
encoder.block.0.layer.1.DenseReluDense.act NewGELUActivation
encoder.block.0.layer.1.layer_norm T5LayerNorm
encoder.block.0.layer.1.dropout Dropout
decoder.block.0.layer ModuleList
decoder.block.0.layer.0 T5LayerSelfAttention

T5ForConditionalGeneration
│
├── Encoder (8 Block)
│   │
│   ├── Layer 0 — SelfAttention
│   │   ├── q
│   │   ├── k
│   │   ├── v
│   │   └── o
│   │
│   └── Layer 1 — DenseReluDense
│       ├── wi_0
│       ├── wi_1
│       └── wo
│
└── Decoder (8 Block)
    │
    ├── Layer 0 — SelfAttention
    │   ├── q
    │   ├── k
    │   ├── v
    │   └── o
    │
    ├── Layer 1 — EncDecAttention
    │   ├── q
    │   ├── k
    │   ├── v
    │   └── o
    │
    └── Layer 2 — DenseReluDense
        ├── wi_0
        ├── wi_1
        └── wo




| Modül ismi | Görevi |
|---|---|
| `SelfAttention` | Aynı dizi içerisindeki token ilişkilerini işler. |
| `EncDecAttention` | Encoder ve decoder arasındaki bilgi aktarımını sağlar. |
| `q` | Query projection |
| `k` | Key projection |
| `v` | Value projection |
| `o` | Output projection |
| `DenseReluDense` | Feed-forward network |
| `wi_0` | Aktivasyon uygulanan giriş projeksiyonu |
| `wi_1` | İkinci giriş projeksiyonu |
| `wo` | Feed-forward çıkış projeksiyonu |
| `relative_attention_bias` | Tokenların göreceli konumlarına ilişkin attention bias |
| `layer_norm` | Token temsillerini normalize eder. |
| `dropout` | Eğitim sırasında overfitting riskini azaltmaya yardımcı olur. |
| `act` | GELU aktivasyon fonksiyonu |


Encoder Self-Attention
Araştırma makalesindeki kelimeler arasındaki ilişkileri işler.
Decoder Self-Attention
Modelin özet üretirken önceki hedef tokenlardan yararlanmasını sağlar. Causal masking sayesinde sonraki tokenları önceden göremez.
Decoder Cross-Attention
Model özet üretirken araştırma makalesinin hangi bilgilerine dikkat etmesi gerektiğini belirler.


In [56]:
import torch.nn as nn

projection_records = []

for name, module in model.named_modules():

    if name.startswith("encoder.block.0.") or name.startswith("decoder.block.0."):

        if isinstance(module, nn.Linear):

            projection_records.append({
                "module_name": name,
                "input_features": module.in_features,
                "output_features": module.out_features,
                "weight_shape": tuple(module.weight.shape),
                "weight_parameters": module.weight.numel()
            })

projection_df = pd.DataFrame(projection_records)

display(projection_df)

,module_name,input_features,output_features,weight_shape,weight_parameters
0,encoder.block.0.layer.0.SelfAttention.q,512,384,"(384, 512)",196608
1,encoder.block.0.layer.0.SelfAttention.k,512,384,"(384, 512)",196608
2,encoder.block.0.layer.0.SelfAttention.v,512,384,"(384, 512)",196608
3,encoder.block.0.layer.0.SelfAttention.o,384,512,"(512, 384)",196608
4,encoder.block.0.layer.1.DenseReluDense.wi_0,512,1024,"(1024, 512)",524288
5,encoder.block.0.layer.1.DenseReluDense.wi_1,512,1024,"(1024, 512)",524288
6,encoder.block.0.layer.1.DenseReluDense.wo,1024,512,"(512, 1024)",524288
7,decoder.block.0.layer.0.SelfAttention.q,512,384,"(384, 512)",196608
8,decoder.block.0.layer.0.SelfAttention.k,512,384,"(384, 512)",196608
9,decoder.block.0.layer.0.SelfAttention.v,512,384,"(384, 512)",196608


In [57]:
ranks = [8, 16]

target_modules = ["q", "v"]

lora_records = []

for rank in ranks:

    lora_parameters = 0
    module_count = 0

    for name, module in model.named_modules():

        if isinstance(module, nn.Linear) and name.split(".")[-1] in target_modules:

            module_count += 1

            lora_parameters += rank * (module.in_features + module.out_features)

    lora_records.append({
        "rank": rank,
        "target_module_count": module_count,
        "lora_parameters": lora_parameters,
        "total_parameters_after_lora": total_parameters + lora_parameters,
        "trainable_ratio": lora_parameters / (total_parameters + lora_parameters) * 100
    })

lora_analysis_df = pd.DataFrame(lora_records)

display(lora_analysis_df)

,rank,target_module_count,lora_parameters,total_parameters_after_lora,trainable_ratio
0,8,48,344064,77305216,0.445072
1,16,48,688128,77649280,0.886200


In [58]:
architecture.update({
    "total_parameters": total_parameters,
    "trainable_parameters": trainable_parameters,
    "frozen_parameters": total_parameters - trainable_parameters,
    "trainable_ratio": trainable_parameters / total_parameters * 100,
    "model_dtype": str(model_dtype),
    "weight_memory_mib": weight_memory_mib
})

architecture_df = pd.DataFrame(
    architecture.items(),
    columns=["feature", "value"]
)

display(architecture_df)

architecture_df.to_csv(
    reports_dir / "model_architecture.csv",
    index=False
)

,feature,value
0,model_architecture,T5ForConditionalGeneration
1,encoder_layers,8
2,decoder_layers,8
3,hidden_size,512
4,attention_heads,6
5,vocabulary_size,32128
6,feed_forward_size,1024
7,key_value_size,64
8,feed_forward_type,gated-gelu
9,total_parameters,76961152


In [59]:
projection_df.to_csv(
    reports_dir / "model_projection_analysis.csv",
    index=False
)

lora_analysis_df.to_csv(
    reports_dir / "lora_parameter_analysis.csv",
    index=False
)

| Precision | Açılımı | Bit | Parametre başına byte |
|---|---|---:|---:|
| FP32 | Floating Point 32 | 32 | 4 |
| FP16 | Floating Point 16 | 16 | 2 |
| BF16 | Brain Floating Point 16 | 16 | 2 |
| INT8 | Integer 8 | 8 | 1 |
| INT4 | Integer 4 | 4 | 0,5 |


| Kavram | Hatırlaman gereken |
|---|---|
| FP32 | Yüksek hassasiyet, yüksek bellek tüketimi |
| FP16 | Daha az bellek, daha dar sayı aralığı |
| BF16 | Daha az bellek, geniş sayı aralığı |
| INT8 | 8 bitlik quantization için kullanılan tam sayı formatı |
| INT4 | 4 bitlik tam sayı formatı |
| Mixed Precision | Farklı işlemlerde farklı sayısal hassasiyetler kullanmak |
| Quantization | Ağırlıkların temsil maliyetini azaltmak |
| LoRA | Eğitilebilir parametre sayısını azaltmak |
| QLoRA | Quantization ve LoRA'yı birleştirmek |

In [60]:
precisions = {
    "FP32": 32,
    "FP16": 16,
    "BF16": 16,
    "INT8": 8,
    "INT4": 4
}

memory_records = []

for precision, bits in precisions.items():

    bytes_per_parameter = bits / 8 

    memory_bytes = total_parameters * bytes_per_parameter

    memory_mib = memory_bytes / (1024 ** 2)

    memory_reduction_rate =  (1 - (memory_mib / weight_memory_mib)) * 100

    memory_records.append({
        "precision": precision,
        "bits": bits,
        "bytes_per_parameter": bytes_per_parameter,
        "total_parameters": total_parameters,
        "weight_memory_mib": memory_mib,
        "memory_reduction_rate": memory_reduction_rate
    })

memory_analysis_df = pd.DataFrame(memory_records)

display(memory_analysis_df)

memory_analysis_df.to_csv(
    reports_dir / "model_memory_analysis.csv",
    index=False
)

print("Model memory analysis saved successfully.")

,precision,bits,bytes_per_parameter,total_parameters,weight_memory_mib,memory_reduction_rate
0,FP32,32,4.0,76961152,293.583496,0.0
1,FP16,16,2.0,76961152,146.791748,50.0
2,BF16,16,2.0,76961152,146.791748,50.0
3,INT8,8,1.0,76961152,73.395874,75.0
4,INT4,4,0.5,76961152,36.697937,87.5


Model memory analysis saved successfully.


Rank gerçekte neyi değiştiriyor?
Rank değerini, LoRA'nın öğrenebileceği ağırlık güncellemelerinin kapasitesini kontrol eden bir hiperparametre olarak düşünebilirsin.

| Özellik | Düşük Rank | Yüksek Rank |
|---|---|---|
| Eğitilebilir parametre | Daha az | Daha fazla |
| Adapter boyutu | Daha küçük | Daha büyük |
| Öğrenme kapasitesi | Daha sınırlı | Daha geniş |
| Adapter bellek maliyeti | Daha düşük | Daha yüksek |
| Underfitting riski | Daha yüksek olabilir | Daha düşük olabilir |
| Overfitting riski | Daha düşük olabilir | Daha yüksek olabilir |

In [61]:
input_features = 768
output_features = 768

ranks = [4, 8, 16, 32]

original_parameters = input_features * output_features

rank_records = []

for rank in ranks:

    a_shape = (rank, input_features)

    b_shape = (output_features, rank)

    lora_parameters = rank * (input_features + output_features)

    trainable_ratio = (lora_parameters / original_parameters) * 100

    reduction_rate = 100 - trainable_ratio

    rank_records.append({
        "rank": rank,
        "a_shape": a_shape,
        "b_shape": b_shape,
        "original_parameters": original_parameters,
        "lora_parameters": lora_parameters,
        "trainable_ratio": trainable_ratio,
        "reduction_rate": reduction_rate
    })

rank_analysis_df = pd.DataFrame(rank_records)

display(rank_analysis_df)

,rank,a_shape,b_shape,original_parameters,lora_parameters,trainable_ratio,reduction_rate
0,4,"(4, 768)","(768, 4)",589824,6144,1.041667,98.958333
1,8,"(8, 768)","(768, 8)",589824,12288,2.083333,97.916667
2,16,"(16, 768)","(768, 16)",589824,24576,4.166667,95.833333
3,32,"(32, 768)","(768, 32)",589824,49152,8.333333,91.666667


In [62]:
%pip install peft

from peft import LoraConfig, get_peft_model, TaskType


lora_config = LoraConfig(
    task_type= TaskType.SEQ_2_SEQ_LM,
    r= 8,
    lora_alpha= 16,
    lora_dropout= 0.05,
    target_modules=["q" , "v"],
    bias="none",
    inference_mode=False
)

display(lora_config)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 886.7/886.7 kB 3.9 MB/s  0:00:00m 24.5 MB/s eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


LoraConfig(task_type=<TaskType.SEQ_2_SEQ_LM: 'SEQ_2_SEQ_LM'>, peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.22.0', base_model_name_or_path=None, revision=None, inference_mode=False, r=8, target_modules={'q', 'v'}, exclude_modules=None, lora_alpha=16, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, astra_config=None, lora_ga_config=None, use_dora=False, velora_config=None, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, monteclora_config=None, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, kasa_config=None, ensure_weight_tying=False)

| Metrik | Full Fine-Tuning | LoRA |
|---|---:|---:|
| Toplam parametre | 76.961.152 | 77.305.216 |
| Eğitilebilir parametre | 76.961.152 | 344.064 |
| Dondurulmuş parametre | 0 | 76.961.152 |
| Eğitilebilir parametre oranı | %100 | %0,4451 |

In [63]:
base_total_parameters = total_parameters

base_trainable_parameters = trainable_parameters

base_frozen_parameters = base_total_parameters - base_trainable_parameters

lora_model = get_peft_model(
    model,
    lora_config
)

lora_model.print_trainable_parameters()

trainable params: 344,064 || all params: 77,305,216 || trainable%: 0.4451


In [66]:
lora_total_parameters = sum(
    p.numel() for p in lora_model.parameters()
)

lora_trainable_parameters = sum(
    p.numel() for p in lora_model.parameters() if p.requires_grad
)

lora_frozen_parameters = sum(
    p.numel() for p in lora_model.parameters() if not p.requires_grad
)

lora_trainable_ratio = (
    lora_trainable_parameters / lora_total_parameters
) * 100

print("Total parameters:", lora_total_parameters)
print("Trainable parameters:", lora_trainable_parameters)
print("Frozen parameters:", lora_frozen_parameters)
print("Trainable ratio:", lora_trainable_ratio)

assert lora_total_parameters == lora_trainable_parameters + lora_frozen_parameters

assert lora_frozen_parameters == base_total_parameters

assert lora_trainable_parameters == 344064

print("Parameter validation: PASS")

Total parameters: 77305216
Trainable parameters: 344064
Frozen parameters: 76961152
Trainable ratio: 0.445072166928555
Parameter validation: PASS


In [67]:
for name, parameter in lora_model.named_parameters():
    if parameter.requires_grad:
        print(name, tuple(parameter.shape))

base_model.model.encoder.block.0.layer.0.SelfAttention.q.lora_A.default.weight (8, 512)
base_model.model.encoder.block.0.layer.0.SelfAttention.q.lora_B.default.weight (384, 8)
base_model.model.encoder.block.0.layer.0.SelfAttention.v.lora_A.default.weight (8, 512)
base_model.model.encoder.block.0.layer.0.SelfAttention.v.lora_B.default.weight (384, 8)
base_model.model.encoder.block.1.layer.0.SelfAttention.q.lora_A.default.weight (8, 512)
base_model.model.encoder.block.1.layer.0.SelfAttention.q.lora_B.default.weight (384, 8)
base_model.model.encoder.block.1.layer.0.SelfAttention.v.lora_A.default.weight (8, 512)
base_model.model.encoder.block.1.layer.0.SelfAttention.v.lora_B.default.weight (384, 8)
base_model.model.encoder.block.2.layer.0.SelfAttention.q.lora_A.default.weight (8, 512)
base_model.model.encoder.block.2.layer.0.SelfAttention.q.lora_B.default.weight (384, 8)
base_model.model.encoder.block.2.layer.0.SelfAttention.v.lora_A.default.weight (8, 512)
base_model.model.encoder.block.2

In [68]:
adapter_records = []

for name, parameter in lora_model.named_parameters():

    if parameter.requires_grad:

        adapter_records.append({
            "parameter_name": name,
            "shape": tuple(parameter.shape),
            "parameter_count": parameter.numel(),
            "trainable": parameter.requires_grad
        })

adapter_df = pd.DataFrame(adapter_records)

display(adapter_df.head(10))

print("Trainable tensors:", len(adapter_df))

print("Total adapter parameters:", adapter_df["parameter_count"].sum())

,parameter_name,shape,parameter_count,trainable
0,base_model.model.encoder.block.0.layer.0.SelfA...,"(8, 512)",4096,True
1,base_model.model.encoder.block.0.layer.0.SelfA...,"(384, 8)",3072,True
2,base_model.model.encoder.block.0.layer.0.SelfA...,"(8, 512)",4096,True
3,base_model.model.encoder.block.0.layer.0.SelfA...,"(384, 8)",3072,True
4,base_model.model.encoder.block.1.layer.0.SelfA...,"(8, 512)",4096,True
5,base_model.model.encoder.block.1.layer.0.SelfA...,"(384, 8)",3072,True
6,base_model.model.encoder.block.1.layer.0.SelfA...,"(8, 512)",4096,True
7,base_model.model.encoder.block.1.layer.0.SelfA...,"(384, 8)",3072,True
8,base_model.model.encoder.block.2.layer.0.SelfA...,"(8, 512)",4096,True
9,base_model.model.encoder.block.2.layer.0.SelfA...,"(384, 8)",3072,True


Trainable tensors: 96
Total adapter parameters: 344064


In [69]:
adapter_df["component"] = adapter_df["parameter_name"].str.extract(
    r"\.(encoder|decoder)\."
)

adapter_df["attention_type"] = adapter_df["parameter_name"].str.extract(
    r"\.(SelfAttention|EncDecAttention)\."
)

adapter_df["projection"] = adapter_df["parameter_name"].str.extract(
    r"\.(q|v)\."
)

adapter_df["matrix_type"] = adapter_df["parameter_name"].str.extract(
    r"lora_(A|B)"
)

adapter_df["module_name"] = adapter_df["parameter_name"].str.split(
    r"\.lora_[AB]\.",
    regex=True
).str[0]

display(adapter_df.head(10))


,parameter_name,shape,parameter_count,trainable,component,attention_type,projection,matrix_type,module_name
0,base_model.model.encoder.block.0.layer.0.SelfA...,"(8, 512)",4096,True,encoder,SelfAttention,q,A,base_model.model.encoder.block.0.layer.0.SelfA...
1,base_model.model.encoder.block.0.layer.0.SelfA...,"(384, 8)",3072,True,encoder,SelfAttention,q,B,base_model.model.encoder.block.0.layer.0.SelfA...
2,base_model.model.encoder.block.0.layer.0.SelfA...,"(8, 512)",4096,True,encoder,SelfAttention,v,A,base_model.model.encoder.block.0.layer.0.SelfA...
3,base_model.model.encoder.block.0.layer.0.SelfA...,"(384, 8)",3072,True,encoder,SelfAttention,v,B,base_model.model.encoder.block.0.layer.0.SelfA...
4,base_model.model.encoder.block.1.layer.0.SelfA...,"(8, 512)",4096,True,encoder,SelfAttention,q,A,base_model.model.encoder.block.1.layer.0.SelfA...
5,base_model.model.encoder.block.1.layer.0.SelfA...,"(384, 8)",3072,True,encoder,SelfAttention,q,B,base_model.model.encoder.block.1.layer.0.SelfA...
6,base_model.model.encoder.block.1.layer.0.SelfA...,"(8, 512)",4096,True,encoder,SelfAttention,v,A,base_model.model.encoder.block.1.layer.0.SelfA...
7,base_model.model.encoder.block.1.layer.0.SelfA...,"(384, 8)",3072,True,encoder,SelfAttention,v,B,base_model.model.encoder.block.1.layer.0.SelfA...
8,base_model.model.encoder.block.2.layer.0.SelfA...,"(8, 512)",4096,True,encoder,SelfAttention,q,A,base_model.model.encoder.block.2.layer.0.SelfA...
9,base_model.model.encoder.block.2.layer.0.SelfA...,"(384, 8)",3072,True,encoder,SelfAttention,q,B,base_model.model.encoder.block.2.layer.0.SelfA...


In [70]:
adapter_distribution_df = adapter_df.groupby(
    ["component", "attention_type"]
).agg(
    adapter_count=("module_name", "nunique"),
    tensor_count=("parameter_name", "count"),
    trainable_parameters=("parameter_count", "sum")
).reset_index()

display(adapter_distribution_df)

,component,attention_type,adapter_count,tensor_count,trainable_parameters
0,decoder,EncDecAttention,16,32,114688
1,decoder,SelfAttention,16,32,114688
2,encoder,SelfAttention,16,32,114688


In [71]:
assert adapter_df[
    ["component", "attention_type", "projection", "matrix_type", "module_name"]
].notna().all().all()

assert adapter_df["module_name"].nunique() == 48

assert len(adapter_df) == 96

assert adapter_df["parameter_count"].sum() == lora_trainable_parameters

assert set(adapter_df["projection"]) == {"q", "v"}

print("Adapter structure validation: PASS")


assert (adapter_df["matrix_type"] == "A").sum() == 48

assert (adapter_df["matrix_type"] == "B").sum() == 48

assert adapter_df.groupby("module_name")["matrix_type"].apply(
    lambda values: sorted(values) == ["A", "B"]
).all()

print("LoRA matrix validation: PASS")


assert adapter_df.loc[
    adapter_df["component"] == "encoder", "module_name"
].nunique() == 16

assert adapter_df.loc[
    adapter_df["component"] == "decoder", "module_name"
].nunique() == 32

print("Encoder-decoder distribution: PASS")


assert all(
    parameter.requires_grad == (
        ".lora_A." in name or ".lora_B." in name
    )
    for name, parameter in lora_model.named_parameters()
)

print("Frozen parameter validation: PASS")


expected_lora_parameters = lora_analysis_df.loc[
    lora_analysis_df["rank"] == lora_config.r,
    "lora_parameters"
].iloc[0]

assert lora_trainable_parameters == expected_lora_parameters

print("Theoretical parameter consistency: PASS")

print("\nAll LoRA configuration tests passed.")

Adapter structure validation: PASS
LoRA matrix validation: PASS
Encoder-decoder distribution: PASS
Frozen parameter validation: PASS
Theoretical parameter consistency: PASS

All LoRA configuration tests passed.


In [72]:
parameter_comparison_df = pd.DataFrame({
    "metric": [
        "total_parameters",
        "trainable_parameters",
        "frozen_parameters",
        "trainable_ratio"
    ],
    "full_fine_tuning": [
        base_total_parameters,
        base_trainable_parameters,
        base_frozen_parameters,
        base_trainable_parameters / base_total_parameters * 100
    ],
    "lora": [
        lora_total_parameters,
        lora_trainable_parameters,
        lora_frozen_parameters,
        lora_trainable_ratio
    ]
})

display(parameter_comparison_df)

lora_analysis_df.to_csv(
    reports_dir / "lora_theoretical_analysis.csv",
    index=False
)

parameter_comparison_df.to_csv(
    reports_dir / "lora_parameter_analysis.csv",
    index=False
)

adapter_df.to_csv(
    reports_dir / "lora_adapter_details.csv",
    index=False
)

adapter_distribution_df.to_csv(
    reports_dir / "lora_adapter_distribution.csv",
    index=False
)

print("All LoRA analysis reports saved successfully.")

,metric,full_fine_tuning,lora
0,total_parameters,76961152.0,7.730522e+07
1,trainable_parameters,76961152.0,3.440640e+05
2,frozen_parameters,0.0,7.696115e+07
3,trainable_ratio,100.0,4.450722e-01


All LoRA analysis reports saved successfully.


| Soru | Cevap |
|---|---|
| LoRA neden toplam parametre sayısını artırır? | Orijinal ağırlıkları silmeden yeni A ve B matrisleri eklediği için. |
| Neden yalnızca adapter parametrelerini güncelliyoruz? | Eğitim maliyetini azaltmak ve temel model ağırlıklarını korumak için. |
| Neden q ve v katmanlarını seçtik? | Attention mekanizmasının bilgi arama ve içerik temsil süreçlerini uyarlamak için. |
| Encoder ve decoder'a neden adapter ekledik? | Hem kaynak metni işleme hem de kaynak metne bağlı özet üretme süreçlerini uyarlamak için. |
| Rank yükseldiğinde ne olur? | Adapter parametreleri, öğrenme kapasitesi ve bellek maliyeti artar. |
| Yüksek rank neden her zaman ROUGE skorunu artırmaz? | Veri kalitesi, overfitting, learning rate ve kaynak metnin yeterliliği gibi başka faktörler de başarıyı etkiler. |

| Özellik | FP32 | FP16 | BF16 | INT8 | NF4 |
|---|---|---|---|---|---|
| Bit | 32 | 16 | 16 | 8 | 4 |
| Byte/parametre | 4 | 2 | 2 | 1 | 0,5 |
| Sayısal format | Float | Float | Float | Integer | NormalFloat |
| Full Fine-Tuning | Evet | Evet* | Evet* | —** | —** |
| LoRA eğitimi | Evet | Evet | Evet | Evet** | Evet** |
| Inference | Evet | Evet | Evet | Evet | Evet |
| Teorik ağırlık belleği | 293,58 MiB | 146,79 MiB | 146,79 MiB | 73,40 MiB | 36,70 MiB |

In [73]:
import torch

cuda_available = torch.cuda.is_available()

hardware_info = {
    "pytorch_version": torch.__version__,
    "cuda_available": cuda_available,
    "cuda_version": torch.version.cuda
}

if cuda_available:

    hardware_info.update({
        "gpu_name": torch.cuda.get_device_name(0),
        "compute_capability": torch.cuda.get_device_capability(0),
        "gpu_memory_gib": torch.cuda.get_device_properties(0).total_memory / (1024 ** 3),
        "bf16_supported": torch.cuda.is_bf16_supported(including_emulation=False)
    })

hardware_df = pd.DataFrame(
    hardware_info.items(),
    columns=["feature", "value"]
)

display(hardware_df)

,feature,value
0,pytorch_version,2.14.1+cu130
1,cuda_available,True
2,cuda_version,13.0
3,gpu_name,NVIDIA GeForce RTX 4060 Laptop GPU
4,compute_capability,"(8, 9)"
5,gpu_memory_gib,7.624573
6,bf16_supported,True


In [77]:
MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 768

USE_BF16 = True
USE_FP16 = False
USE_QUANTIZATION = False

sample = prepared_dataset["train"][0]

source_text = sample["input_text"]

print(source_text)


source_tokens = tokenizer(
    source_text,
    max_length=MAX_INPUT_LENGTH,
    truncation=True,
    padding=False
)

print(source_tokens.keys())

print("Input length:", len(source_tokens["input_ids"]))

print("Attention mask length:", len(source_tokens["attention_mask"]))

print("First 20 tokens:", source_tokens["input_ids"][:20])

Summarize the following biomedical research article in plain language for a general audience:

How long neural information is stored in a local brain area reflects functions of that region and is often estimated by the magnitude of the autocorrelation of intrinsic neural signals in the area . Here , we investigated such intrinsic neural timescales in high-functioning adults with autism and examined whether local brain dynamics reflected their atypical behaviours . By analysing resting-state fMRI data , we identified shorter neural timescales in the sensory/visual cortices and a longer timescale in the right caudate in autism . The shorter intrinsic timescales in the sensory/visual areas were correlated with the severity of autism , whereas the longer timescale in the caudate was associated with cognitive rigidity . These observations were confirmed from neurodevelopmental perspectives and replicated in two independent cross-sectional datasets . Moreover , the intrinsic timescale was co

In [78]:
target_text = sample["target_summary"]

print(target_text)

Autism is a brain disorder that affects how people interact with others . It occupies a spectrum , with severe autism at one end and high-functioning autism at the other . People with severe autism usually have intellectual impairments and little spoken language . Those with high-functioning autism have average or above average IQ , but struggle with more subtle aspects of communication , such as body language . As well as social difficulties , many individuals with autism show repetitive behaviors and have narrow interests . The brains of people with autism process information differently to those of people without autism . The brain as a whole shows less coordinated activity in autism , for example . But whether individual brain regions themselves also work differently in autism is unclear . Watanabe et al . set out to answer this question by using a brain scanner to compare the resting brain activity of high-functioning people with autism to that of people without autism . In both g

In [79]:
target_tokens = tokenizer(
    text_target=target_text,
    max_length=MAX_TARGET_LENGTH,
    truncation=True,
    padding=False
)

print("Target length:", len(target_tokens["input_ids"]))

print("First 20 target tokens:", target_tokens["input_ids"][:20])

Target length: 466
First 20 target tokens: [27777, 19, 3, 9, 2241, 9311, 24, 2603, 7, 149, 151, 6815, 28, 717, 3, 5, 94, 28979, 7, 3]


In [81]:
training_sample = {
    "input_ids": source_tokens["input_ids"],
    "attention_mask": source_tokens["attention_mask"],
    "labels": target_tokens["input_ids"]
}

for key, value in training_sample.items():
    print(key, len(value))

input_ids 512
attention_mask 512
labels 466


In [84]:
def tokenize_function(examples):

    model_inputs = tokenizer(
        examples["input_text"],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding=False
    )

    target_tokens = tokenizer(
        text_target=examples["target_summary"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding=False
    )

    model_inputs["labels"] = target_tokens["input_ids"]

    return model_inputs


tokenized_sample = tokenize_function(
    prepared_dataset["train"][0]
)

for key, value in tokenized_sample.items():
    print(key, len(value))

assert dict(tokenized_sample) == training_sample

print("Tokenization validation: PASS")

input_ids 512
attention_mask 512
labels 466
Tokenization validation: PASS


In [85]:
tokenized_dataset = prepared_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=prepared_dataset["train"].column_names
)

display(tokenized_dataset)

Map: 100%|██████████| 100/100 [00:00<00:00, 3393.01 examples/s]


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 500
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 100
    })
})

In [86]:
train_source_lengths = [
    len(x) for x in tokenized_dataset["train"]["input_ids"]
]

train_target_lengths = [
    len(x) for x in tokenized_dataset["train"]["labels"]
]

print("Source min:", min(train_source_lengths))
print("Source max:", max(train_source_lengths))
print("Source mean:", np.mean(train_source_lengths))

print("Target min:", min(train_target_lengths))
print("Target max:", max(train_target_lengths))
print("Target mean:", np.mean(train_target_lengths))

Source min: 511
Source max: 512
Source mean: 511.898
Target min: 258
Target max: 768
Target mean: 521.33


In [89]:
validation_source_lengths = [
    len(x) for x in tokenized_dataset["validation"]["input_ids"]
]

validation_target_lengths = [
    len(x) for x in tokenized_dataset["validation"]["labels"]
]

print("Source min:", min(validation_source_lengths))
print("Source max:", max(validation_source_lengths))
print("Source mean:", np.mean(validation_source_lengths))

print("Target min:", min(validation_target_lengths))
print("Target max:", max(validation_target_lengths))
print("Target mean:", np.mean(validation_target_lengths))

Source min: 511
Source max: 512
Source mean: 511.89
Target min: 302
Target max: 759
Target mean: 521.41


In [92]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=lora_model,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt"
)

batch_samples = [
    tokenized_dataset["train"][i] for i in range(4)
]

for sample in batch_samples:
    print(
        "Source:", len(sample["input_ids"]),
        "Target:", len(sample["labels"])
    )

Source: 512 Target: 466
Source: 512 Target: 349
Source: 512 Target: 346
Source: 512 Target: 625


In [94]:
batch = data_collator(batch_samples)


for key, value in batch.items():
    print(key, value.shape)

input_ids torch.Size([4, 512])
attention_mask torch.Size([4, 512])
labels torch.Size([4, 625])
decoder_input_ids torch.Size([4, 625])


In [96]:
print("Labels:")
print(batch["labels"][0, :15])

print("Decoder input IDs:")
print(batch["decoder_input_ids"][0, :15])

print("Last 10 label tokens:")
print(batch["labels"][0, -10:])

print("Last 10 decoder tokens:")
print(batch["decoder_input_ids"][0, -10:])

padding_count = (batch["labels"] == -100).sum().item()

print("Total ignored label tokens:", padding_count)

Labels:
tensor([27777,    19,     3,     9,  2241,  9311,    24,  2603,     7,   149,
          151,  6815,    28,   717,     3])
Decoder input IDs:
tensor([    0, 27777,    19,     3,     9,  2241,  9311,    24,  2603,     7,
          149,   151,  6815,    28,   717])
Last 10 label tokens:
tensor([-100, -100, -100, -100, -100, -100, -100, -100, -100, -100])
Last 10 decoder tokens:
tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
Total ignored label tokens: 714


In [97]:
TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
NUM_EPOCHS = 3
LEARNING_RATE = 2e-4
SEED = 42
NUM_GPUS = 1

effective_batch_size = TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS * NUM_GPUS

print("Train batch size:", TRAIN_BATCH_SIZE)
print("Gradient accumulation:", GRADIENT_ACCUMULATION_STEPS)
print("Effective batch size:", effective_batch_size)

Train batch size: 4
Gradient accumulation: 4
Effective batch size: 16


In [98]:
import math

train_sample_count = len(tokenized_dataset["train"])

batches_per_epoch = math.ceil(
    train_sample_count / TRAIN_BATCH_SIZE
)

optimizer_steps_per_epoch = math.ceil(
    batches_per_epoch / GRADIENT_ACCUMULATION_STEPS
)

total_optimizer_steps = optimizer_steps_per_epoch * NUM_EPOCHS

total_micro_batches = batches_per_epoch * NUM_EPOCHS

print("Train samples:", train_sample_count)
print("Batches per epoch:", batches_per_epoch)
print("Optimizer steps per epoch:", optimizer_steps_per_epoch)
print("Total optimizer steps:", total_optimizer_steps)
print("Total micro batches:", total_micro_batches)

Train samples: 500
Batches per epoch: 125
Optimizer steps per epoch: 32
Total optimizer steps: 96
Total micro batches: 375


In [99]:
training_configuration = {
    "model_name": MODEL_NAME,
    "fine_tuning_method": "LoRA",
    "task_type": "SEQ_2_SEQ_LM",
    "lora_rank": lora_config.r,
    "lora_alpha": lora_config.lora_alpha,
    "lora_dropout": lora_config.lora_dropout,
    "target_modules": sorted(lora_config.target_modules),
    "bias": lora_config.bias,
    "epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "effective_batch_size": effective_batch_size,
    "max_input_length": MAX_INPUT_LENGTH,
    "max_target_length": MAX_TARGET_LENGTH,
    "bf16": USE_BF16,
    "fp16": USE_FP16,
    "quantization": USE_QUANTIZATION,
    "dynamic_padding": True,
    "label_pad_token_id": -100,
    "seed": SEED,
    "train_samples": len(tokenized_dataset["train"]),
    "validation_samples": len(tokenized_dataset["validation"]),
    "total_parameters": lora_total_parameters,
    "trainable_parameters": lora_trainable_parameters,
    "frozen_parameters": lora_frozen_parameters,
    "trainable_ratio": lora_trainable_ratio,
    "batches_per_epoch": batches_per_epoch,
    "optimizer_steps_per_epoch": optimizer_steps_per_epoch,
    "total_optimizer_steps": total_optimizer_steps,
    "total_micro_batches": total_micro_batches,
    "evaluation_metric": "ROUGE",
    "model_selection_metric": "rougeL",
    "greater_is_better": True
}

training_configuration_df = pd.DataFrame(
    training_configuration.items(),
    columns=["parameter", "value"]
)

display(training_configuration_df)

training_configuration_df.to_csv(
    reports_dir / "training_configuration.csv",
    index=False
)

print("Training configuration saved successfully.")

,parameter,value
0,model_name,google/flan-t5-small
1,fine_tuning_method,LoRA
2,task_type,SEQ_2_SEQ_LM
3,lora_rank,8
4,lora_alpha,16
5,lora_dropout,0.05
6,target_modules,"[q, v]"
7,bias,none
8,epochs,3
9,learning_rate,0.0002


Training configuration saved successfully.


In [100]:
tokenized_dataset.save_to_disk(
    str(project_dir / "data" / "healthbrief_tokenized")
)

print("Tokenized dataset saved successfully.")

Saving the dataset (1/1 shards): 100%|██████████| 100/100 [00:00<00:00, 27328.02 examples/s]

Tokenized dataset saved successfully.


In [101]:
assert set(tokenized_dataset.keys()) == {"train", "validation"}

assert len(tokenized_dataset["train"]) == 500
assert len(tokenized_dataset["validation"]) == 100

for split in ["train", "validation"]:

    assert set(tokenized_dataset[split].column_names) == {
        "input_ids", "attention_mask", "labels"
    }

    for sample in tokenized_dataset[split]:

        assert 0 < len(sample["input_ids"]) <= MAX_INPUT_LENGTH

        assert 0 < len(sample["labels"]) <= MAX_TARGET_LENGTH

        assert len(sample["input_ids"]) == len(sample["attention_mask"])

        assert -100 not in sample["labels"]

    print(f"{split} tokenization validation: PASS")

assert batch["input_ids"].shape == (4, 512)

assert batch["labels"].shape == (4, 625)

assert batch["labels"].shape == batch["decoder_input_ids"].shape

assert padding_count == 714

print("Data collator validation: PASS")

print("\nAll training preparation tests passed.")

train tokenization validation: PASS
validation tokenization validation: PASS
Data collator validation: PASS

All training preparation tests passed.


In [102]:
from datasets import load_from_disk

reloaded_tokenized_dataset = load_from_disk(
    str(project_dir / "data" / "healthbrief_tokenized")
)

for split in ["train", "validation"]:
    assert reloaded_tokenized_dataset[split].to_pandas().equals(
        tokenized_dataset[split].to_pandas()
    )

print("Saved tokenized dataset validation: PASS")

Saved tokenized dataset validation: PASS
